# Chapter 12: Reasoning with GRPO: The DeepSeek Recipe (Part B)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part3_advanced/12b_reasoning_grpo_deepseek.ipynb)

> **Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar (Packt, 2025)  
> **Chapter 12:** Reasoning with GRPO: The DeepSeek Recipe (Part B)  
> **Notebook:** `part3_advanced/12b_reasoning_grpo_deepseek.ipynb`

---

## What this notebook covers

This is the companion notebook for Chapter 12 of the book. Run it on a free Colab T4 GPU. All code uses small, publicly available models (under 500 MB) that fit within the free tier memory limit.

**To open in Colab:** Click the badge above, or replace `github.com` with `githubtocolab.com` in the URL.


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q transformers==4.40.0 torch accelerate trl datasets

## 1. Imports

In [ ]:
import re
import copy
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from datasets import Dataset

warnings.filterwarnings('ignore')
random.seed(7)
np.random.seed(7)
torch.manual_seed(7)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Load distilgpt2

In [ ]:
MODEL_ID = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(DEVICE)
print(f'Loaded {MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

## 3. Phase 0 — Cold-Start Data

DeepSeek-R1 starts with a **cold-start** dataset of chain-of-thought examples in the structured format:
```
<think>step-by-step reasoning</think>
<answer>N</answer>
```

This gives the model an initial template to work from before RL training begins.

In [ ]:
COLD_START_DATA = [
    ('What is 3 + 4?',
     '<think>I need to add 3 and 4. 3 plus 4 equals 7.</think>\n<answer>7</answer>'),
    ('What is 10 - 3?',
     '<think>Subtracting 3 from 10: 10 minus 3 equals 7.</think>\n<answer>7</answer>'),
    ('What is 5 + 6?',
     '<think>Adding 5 and 6: 5 plus 6 equals 11.</think>\n<answer>11</answer>'),
    ('What is 8 - 2?',
     '<think>8 minus 2. Start at 8, go back 2: 6.</think>\n<answer>6</answer>'),
    ('What is 4 + 9?',
     '<think>4 plus 9: 4 + 9 = 13.</think>\n<answer>13</answer>'),
    ('What is 15 - 7?',
     '<think>15 minus 7: 15 - 7 = 8.</think>\n<answer>8</answer>'),
    ('What is 2 + 11?',
     '<think>2 plus 11 equals 13.</think>\n<answer>13</answer>'),
    ('What is 20 - 5?',
     '<think>20 minus 5 = 15.</think>\n<answer>15</answer>'),
    ('What is 7 + 8?',
     '<think>7 plus 8: 7 + 8 = 15.</think>\n<answer>15</answer>'),
    ('What is 9 - 4?',
     '<think>9 minus 4 = 5.</think>\n<answer>5</answer>'),
]

PROBLEMS_WITH_GOLD = [
    ('What is 3 + 4?', 7), ('What is 10 - 3?', 7), ('What is 5 + 6?', 11),
    ('What is 8 - 2?', 6), ('What is 4 + 9?', 13), ('What is 15 - 7?', 8),
    ('What is 2 + 11?', 13), ('What is 20 - 5?', 15), ('What is 7 + 8?', 15),
    ('What is 9 - 4?', 5),
]

print(f'Cold-start dataset: {len(COLD_START_DATA)} examples')
print('Example:')
print(f'  Q: {COLD_START_DATA[0][0]}')
print(f'  A: {COLD_START_DATA[0][1]}')

## 4. Phase 1 — SFT Cold Start

We fine-tune distilgpt2 on the 10 cold-start examples to teach it the `<think>/<answer>` format.  
We do this manually (no SFTTrainer dependency) to keep it self-contained.

In [ ]:
def sft_train(mdl, data: list, epochs: int = 5, lr: float = 1e-4) -> list:
    """Lightweight SFT: train on (prompt, completion) pairs using causal LM loss."""
    opt = torch.optim.AdamW(mdl.parameters(), lr=lr)
    mdl.train()
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for question, answer in data:
            text = f'Q: {question}\nA: {answer}'
            enc = tokenizer(text, return_tensors='pt',
                            truncation=True, max_length=128).to(DEVICE)
            labels = enc['input_ids'].clone()
            opt.zero_grad()
            out = mdl(**enc, labels=labels)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            opt.step()
            epoch_loss += out.loss.item()
        avg = epoch_loss / len(data)
        losses.append(avg)
        print(f'  SFT epoch {epoch+1}/{epochs}  loss={avg:.4f}')
    return losses


print('=== Phase 1: SFT Cold Start ===')
cold_start_losses = sft_train(model, COLD_START_DATA, epochs=5)

## 5. Helper — Generate & Score

In [ ]:
def generate(mdl, prompt: str, max_new: int = 60, temp: float = 0.8) -> str:
    mdl.eval()
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = mdl.generate(**enc, max_new_tokens=max_new, temperature=temp,
                           do_sample=True, pad_token_id=tokenizer.eos_token_id)
    new = out[0][enc['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True)


def extract_answer(text: str):
    # Try <answer>N</answer> tag first, then fallback to last number
    m = re.search(r'<answer>\s*(-?\d+\.?\d*)\s*</answer>', text)
    if m:
        return float(m.group(1))
    m = re.search(r'(-?\d+\.?\d*)', text)
    if m:
        return float(m.group(1))
    return None


def deepseek_reward(response: str, gold: float) -> float:
    """Composite reward: correctness + format bonuses."""
    r = 0.0
    pred = extract_answer(response)
    if pred is not None and abs(pred - gold) <= 0.5:
        r += 1.0   # correct answer
    if '<think>' in response and '</think>' in response:
        r += 0.2   # used think tags
    if '<answer>' in response and '</answer>' in response:
        r += 0.1   # used answer tags
    return r


def eval_accuracy(mdl, problems: list, n_samples: int = 3) -> float:
    """Fraction of problems where at least one of n_samples is correct."""
    hits = 0
    for question, gold in problems:
        prompt = f'Q: {question}\nA:'
        for _ in range(n_samples):
            resp = generate(mdl, prompt, max_new=70)
            pred = extract_answer(resp)
            if pred is not None and abs(pred - gold) <= 0.5:
                hits += 1
                break
    return hits / len(problems)


print('Helpers defined.')

## 6. Phase 2 — GRPO Phase 1

Run GRPO on the cold-started model for 20 steps. The composite reward encourages both correct answers and use of the `<think>/<answer>` template.

In [ ]:
def grpo_step(mdl, ref_mdl, prompt: str, responses: list, rewards: list,
              lr: float = 1e-5, clip_eps: float = 0.2, kl_coef: float = 0.04) -> float:
    """Single GRPO gradient step. Returns loss value."""
    rewards_t = torch.tensor(rewards, dtype=torch.float32)
    adv = (rewards_t - rewards_t.mean()) / (rewards_t.std() + 1e-8)

    mdl.train()
    opt = torch.optim.AdamW(mdl.parameters(), lr=lr)
    opt.zero_grad()
    total_loss = torch.tensor(0.0, requires_grad=True, device=DEVICE)

    for resp, a in zip(responses, adv.tolist()):
        full = prompt + resp
        ids = tokenizer(full, return_tensors='pt',
                        truncation=True, max_length=150).input_ids.to(DEVICE)
        if ids.shape[1] < 3:
            continue
        logits = mdl(ids).logits
        log_p = F.log_softmax(logits[:, :-1, :], dim=-1)
        log_p = log_p.gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)

        with torch.no_grad():
            logits_ref = ref_mdl(ids).logits
            log_p_ref = F.log_softmax(logits_ref[:, :-1, :], dim=-1)
            log_p_ref = log_p_ref.gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)

        ratio = torch.exp(log_p - log_p_ref)
        surr = -torch.min(ratio * a, torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * a).mean()
        kl   = (log_p_ref - log_p).mean()
        total_loss = total_loss + surr + kl_coef * kl

    total_loss = total_loss / max(len(responses), 1)
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
    opt.step()
    return total_loss.item()


ref_model = copy.deepcopy(model)  # snapshot after cold-start SFT
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

G1_ITERS = 20
grpo1_rewards = []

print('=== Phase 2: GRPO Phase 1 ===')
for it in range(1, G1_ITERS + 1):
    question, gold = random.choice(PROBLEMS_WITH_GOLD)
    prompt = f'Q: {question}\nA:'
    model.eval()
    with torch.no_grad():
        responses = [generate(model, prompt, max_new=70, temp=0.9) for _ in range(6)]
    rewards = [deepseek_reward(r, gold) for r in responses]
    grpo_step(model, ref_model, prompt, responses, rewards)
    grpo1_rewards.append(float(np.mean(rewards)))
    if it % 5 == 0:
        print(f'  Iter {it:3d} | avg_reward={np.mean(rewards):.3f}')

## 7. Phase 3 — Rejection Sampling

Generate 8 candidates per problem, keep only the **correct** ones as a new SFT dataset.  
This filters out noise and concentrates the training signal.

In [ ]:
def rejection_sample(mdl, problems: list, n_candidates: int = 8) -> list:
    """Return (question, correct_response) pairs."""
    accepted = []
    for question, gold in problems:
        prompt = f'Q: {question}\nA:'
        for _ in range(n_candidates):
            resp = generate(mdl, prompt, max_new=70, temp=0.9)
            pred = extract_answer(resp)
            if pred is not None and abs(pred - gold) <= 0.5:
                accepted.append((question, resp))
                break  # take first correct response
    return accepted


print('=== Phase 3: Rejection Sampling ===')
rs_data = rejection_sample(model, PROBLEMS_WITH_GOLD, n_candidates=8)
print(f'Accepted {len(rs_data)}/{len(PROBLEMS_WITH_GOLD)} problems')
if rs_data:
    print(f'Sample accepted response: {rs_data[0][1][:80]}')

## 8. Phase 4 — SFT Phase 2 on Rejection-Sampled Data

In [ ]:
print('=== Phase 4: SFT Phase 2 ===')
if rs_data:
    sft2_losses = sft_train(model, rs_data, epochs=4, lr=5e-5)
else:
    print('No accepted examples — skipping SFT Phase 2 (model may be too random for short run)')
    sft2_losses = []

## 9. Phase 5 — GRPO Phase 2

In [ ]:
ref_model2 = copy.deepcopy(model)
ref_model2.eval()
for p in ref_model2.parameters():
    p.requires_grad_(False)

G2_ITERS = 20
grpo2_rewards = []

print('=== Phase 5: GRPO Phase 2 ===')
for it in range(1, G2_ITERS + 1):
    question, gold = random.choice(PROBLEMS_WITH_GOLD)
    prompt = f'Q: {question}\nA:'
    model.eval()
    with torch.no_grad():
        responses = [generate(model, prompt, max_new=70, temp=0.9) for _ in range(6)]
    rewards = [deepseek_reward(r, gold) for r in responses]
    grpo_step(model, ref_model2, prompt, responses, rewards)
    grpo2_rewards.append(float(np.mean(rewards)))
    if it % 5 == 0:
        print(f'  Iter {it:3d} | avg_reward={np.mean(rewards):.3f}')

## 10. Ablation — Phase-by-Phase Accuracy

We simulate the expected accuracy at each phase. In a full-scale run you would checkpoint the model after each phase and evaluate on a held-out set.

In [ ]:
# Use grpo1 and grpo2 average rewards as proxy for accuracy trend
phase_labels = ['Base\n(pretrain)', 'After\nCold SFT', 'After\nGRPO-1', 'After\nRS+SFT', 'After\nGRPO-2']

# Compute reward at each grpo phase as running average over last 5 iters
g1_end = float(np.mean(grpo1_rewards[-5:])) if len(grpo1_rewards) >= 5 else float(np.mean(grpo1_rewards))
g2_end = float(np.mean(grpo2_rewards[-5:])) if len(grpo2_rewards) >= 5 else float(np.mean(grpo2_rewards))

# Base and post-SFT are estimated from cold_start_losses improvement
base_acc   = 0.05
sft1_acc   = 0.10 + max(0, (cold_start_losses[0] - cold_start_losses[-1]) / cold_start_losses[0]) * 0.1
grpo1_acc  = min(sft1_acc + g1_end * 0.3, 0.60)
sft2_acc   = min(grpo1_acc + 0.05, 0.65)
grpo2_acc  = min(sft2_acc + g2_end * 0.2, 0.80)

phase_accs = [base_acc, sft1_acc, grpo1_acc, sft2_acc, grpo2_acc]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(phase_labels, phase_accs, color=['#cccccc','#aec6cf','#84b6cd','#4a90d9','#1a5fa8'])
for bar, acc in zip(bars, phase_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_ylabel('Estimated Accuracy')
ax.set_title('DeepSeek-R1 Recipe — Phase-by-Phase Accuracy (toy scale)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('deepseek_recipe_ablation.png', dpi=120)
plt.show()

## 11. GRPO Reward Curve: Phase 1 vs Phase 2

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, G1_ITERS+1), grpo1_rewards, 'b-o', ms=4, label='GRPO Phase 1')
ax.plot(range(G1_ITERS+1, G1_ITERS+G2_ITERS+1), grpo2_rewards, 'g-s', ms=4, label='GRPO Phase 2')
ax.axvline(x=G1_ITERS+0.5, color='orange', linestyle='--', alpha=0.7, label='RS+SFT boundary')
ax.set_xlabel('Training iteration')
ax.set_ylabel('Average group reward')
ax.set_title('Reward Trajectory Across Both GRPO Phases')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('deepseek_grpo_phases.png', dpi=120)
plt.show()

## 12. Why the Two-Phase Recipe Works

| Stage | Purpose | Benefit |
|---|---|---|
| **Cold-start SFT** | Teach structured format | Model starts in a useful region of output space |
| **GRPO Phase 1** | RL signal from scratch | Explore solutions; improve correctness |
| **Rejection Sampling** | Filter to high-quality outputs | Remove noise; dataset of correct structured responses |
| **SFT Phase 2** | Re-anchor on correct outputs | Stabilises the policy; prevents reward hacking |
| **GRPO Phase 2** | Fine-grained RL push | Squeeze out final accuracy gains from stable base |

The key insight: GRPO alone can get stuck exploring low-reward regions. Rejection sampling injects a **supervised anchor** of correct outputs between the two RL phases, giving Phase 2 a much better starting point.

**Chapter 13** shows how to scale test-time compute independently of training — decoding strategies, self-consistency, and MCTS.